In [1]:
import duckdb
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor

In [3]:
import pandas as pd

df = pd.read_csv("billing_availability.csv")
gaps = df[(df.status != 200) & (df.date >= "2022-01-01") & (df.date <= "2025-12-31")]
print(gaps)

            date  status
1699  2022-08-27     404
2506  2024-11-11     404


In [3]:
con = duckdb.connect()  # in-memory, or duckdb.connect("local.db") to persist
con.sql("INSTALL httpfs; LOAD httpfs;")

# basic data retrieval test to check if out-the-box connection has any issues 
con.sql("""
    SELECT *
    FROM read_csv_auto('https://u1pfm.upcloudobjects.com/dwh-fina/billing/year=2024/month=01/day=01/billing.csv')
    limit 10

""").show()

┌─────────────────────┬──────────────────────────────────────┬─────────┬──────────────┬─────────┬──────────────┬────────────────┬─────────┬───────────────┬──────────────────────────────────────┬──────────┬───────┬─────────┬─────────┐
│      timestamp      │             resource_id              │ user_id │ credit_usage │ region  │ service_tier │ operation_type │ success │ resource_type │              invoice_id              │ currency │ year  │  month  │   day   │
│      timestamp      │               varchar                │  int64  │    double    │ varchar │   varchar    │    varchar     │ boolean │    varchar    │               varchar                │ varchar  │ int64 │ varchar │ varchar │
├─────────────────────┼──────────────────────────────────────┼─────────┼──────────────┼─────────┼──────────────┼────────────────┼─────────┼───────────────┼──────────────────────────────────────┼──────────┼───────┼─────────┼─────────┤
│ 2024-01-01 16:19:39 │ 282be4c5-3b1f-4b98-87ee-6b06a7ba9760 │  

In [7]:
con = duckdb.connect()  # in-memory, or duckdb.connect("local.db") to persist
con.sql("INSTALL httpfs; LOAD httpfs;")

# basic data retrieval test to check if out-the-box connection has any issues 
con.sql("""
    SELECT success, count(*)
    FROM read_csv_auto('https://u1pfm.upcloudobjects.com/dwh-fina/billing/year=2024/month=01/day=01/billing.csv')
    group by all 

""").show()

# findings 
# resource ID is unique 
# invoice ID is also unique

┌─────────┬──────────────┐
│ success │ count_star() │
│ boolean │    int64     │
├─────────┼──────────────┤
│ false   │        52114 │
│ true    │       979155 │
└─────────┴──────────────┘



In [16]:
url = "https://u1pfm.upcloudobjects.com/dwh-fina/billing/year=2024/month=01/day=01/billing.csv"
r = requests.head(url)
print(r.status_code)

# 200: reads work anonymously. I can build the URLs myself and check which ones exist

200


In [27]:
# cheking out data completeness 
df["date"] = pd.to_datetime(df["date"])
found = df[df["status"] == 200]

counts = (
    found.groupby(
        [
            found["date"].dt.year.rename("year"),
            found["date"].dt.month.rename("month")
        ]
    )
    .size()
    .reset_index(name="n_files")
)

counts

,year,month,n_files
0,2022,1,31
1,2022,2,28
2,2022,3,31
3,2022,4,30
4,2022,5,31
5,2022,6,30
6,2022,7,31
7,2022,8,30
8,2022,9,30
9,2022,10,31
